# Regresion Logistica Binaria

## Objetivo
Implementar un modelo de Regresion Logistica Binaria para predecir la direccion del cambio
de tasas de cambio (sube = 1, baja/estable = 0).

In [ ]:
# Celda 1: Configuracion del entorno
import sys
import os

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from dotenv import load_dotenv
load_dotenv(os.path.join(PROJECT_ROOT, ".env.local"))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Celda 2: Carga de datos
import psycopg2

def get_connection():
    db_host = os.getenv("DB_HOST", "postgres")
    
    if db_host == "postgres":
        host = "postgres"
        port = os.getenv("DB_PORT", "5432")
    else:
        host = "localhost"
        port = os.getenv("DB_PORT", "5433")
    return psycopg2.connect(
        host=host,
        port=port,
        dbname=os.getenv("DB_NAME", "exchange_db"),
        user=os.getenv("DB_USER", "postgres"),
        password=os.getenv("DB_PASSWORD", "postgres"),
        sslmode="disable"
    )

conn = get_connection()

query = """
SELECT 
    er.id,
    c1.code AS base_currency,
    c2.code AS target_currency,
    er.rate,
    er.timestamp
FROM exchange_rates er
JOIN currencies c1 ON er.base_currency_id = c1.id
JOIN currencies c2 ON er.target_currency_id = c2.id
ORDER BY er.timestamp;
"""

df = pd.read_sql(query, conn)
conn.close()
df["timestamp"] = pd.to_datetime(df["timestamp"])
df.head()

## Preparacion de Datos

In [ ]:
# Celda 3: Crear variable objetivo binaria
df_model = df.copy()
df_model = df_model.sort_values(['target_currency', 'timestamp'])

df_model['rate_lag1'] = df_model.groupby('target_currency')['rate'].shift(1)
df_model['rate_lag2'] = df_model.groupby('target_currency')['rate'].shift(2)
df_model['rate_lag3'] = df_model.groupby('target_currency')['rate'].shift(3)
df_model['rate_lag5'] = df_model.groupby('target_currency')['rate'].shift(5)

df_model['rate_change'] = df_model['rate'] - df_model['rate_lag1']
df_model['pct_change'] = df_model['rate_change'] / df_model['rate_lag1'] * 100

df_model['rolling_mean_3'] = df_model.groupby('target_currency')['rate'].transform(lambda x: x.rolling(3).mean())
df_model['rolling_std_3'] = df_model.groupby('target_currency')['rate'].transform(lambda x: x.rolling(3).std())
df_model['rolling_mean_5'] = df_model.groupby('target_currency')['rate'].transform(lambda x: x.rolling(5).mean())

df_model['volatility'] = df_model['rolling_std_3'] / df_model['rolling_mean_3'] * 100

df_model['direction'] = (df_model['rate_change'] > 0).astype(int)

df_model = df_model.dropna(subset=['rate_lag1', 'rate_change'])
df_model = df_model.fillna(0)
print(f"Forma del dataset: {df_model.shape}")
print(f"Distribucion de la variable objetivo:\n{df_model['direction'].value_counts()}")

## Entrenamiento del Modelo

In [ ]:
# Celda 4: Division de datos
features = ['rate_lag1', 'rate_lag2', 'rate_lag3', 'rate_lag5',
            'pct_change', 'rolling_mean_3', 'rolling_std_3', 'rolling_mean_5', 'volatility']

X = df_model[features]
y = df_model['direction']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Entrenamiento: {X_train.shape[0]} | Prueba: {X_test.shape[0]}")
print(f"Clase 1 en train: {y_train.sum()} ({y_train.mean()*100:.1f}%)")
print(f"Clase 1 en test: {y_test.sum()} ({y_test.mean()*100:.1f}%)")

In [ ]:
# Celda 5: Entrenamiento del modelo
log_reg = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight='balanced'
)

log_reg.fit(X_train, y_train)

y_pred = log_reg.predict(X_test)
y_prob = log_reg.predict_proba(X_test)[:, 1]

print("Metricas del Modelo:")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall: {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score: {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")

## Matriz de Confusion y Clasificacion

In [ ]:
# Celda 6: Matriz de Confusion
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation='nearest', cmap='Blues')
plt.title('Matriz de Confusion')
plt.colorbar()

classes = ['Baja/Estable (0)', 'Sube (1)']
tick_marks = np.arange(len(classes))
plt.xticks(tick_marks, classes)
plt.yticks(tick_marks, classes)

for i in range(2):
    for j in range(2):
        plt.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=12)

plt.ylabel('Real')
plt.xlabel('Predicho')
plt.tight_layout()
plt.show()

print("\nReporte de Clasificacion:")
print(classification_report(y_test, y_pred, target_names=classes))

## Validacion Cruzada

In [ ]:
# Celda 7: Cross-Validation
cv_scores = cross_val_score(log_reg, X, y, cv=5, scoring='accuracy')

print("Validacion Cruzada (5 folds):")
print(f"Accuracy por fold: {cv_scores}")
print(f"Accuracy promedio: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

## Analisis de Coeficientes

In [ ]:
# Celda 8: Coeficientes del modelo
coeff_df = pd.DataFrame({
    'feature': features,
    'coefficient': log_reg.coef_[0],
    'odds_ratio': np.exp(log_reg.coef_[0])
}).sort_values('coefficient', key=abs, ascending=False)

print("Coeficientes del modelo (ordenados por magnitud):")
coeff_df

## Curva ROC

In [ ]:
# Celda 9: Curva ROC
from sklearn.metrics import roc_curve

fpr, tpr, thresholds = roc_curve(y_test, y_prob)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label='Curva ROC')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Clasificador Aleatorio')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Tasa de Falsos Positivos')
plt.ylabel('Tasa de Verdaderos Positivos')
plt.title('Curva ROC - Regresion Logistica')
plt.legend(loc='lower right')
plt.grid(True)
plt.tight_layout()
plt.show()

## Conclusion

In [ ]:
# Celda 10: Resumen
resultado = pd.DataFrame({
    'Metrica': ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC'],
    'Valor': [
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred),
        recall_score(y_test, y_pred),
        f1_score(y_test, y_pred),
        roc_auc_score(y_test, y_prob)
    ]
})
resultado

In [ ]:
# Celda 11: Guardar modelo
import joblib

joblib.dump(log_reg, 'regresion_logistica_binaria_model.pkl')
print("Modelo guardado: regresionlogisticabinariamodel.pkl")